<a href="https://colab.research.google.com/github/hermonj-spec/TransactionFraudDetection/blob/main/Online_Transaction_Fraud_Detection_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

df = pd.read_csv("fraud_detection_dataset_10000.csv")
df.head()

,transaction_amount,transaction_hour,account_age_days,transactions_last_24h,location_change,device_new,previous_fraud_count,fraud
0,93.85,22,1616,1,0,1,0,0
1,602.02,3,1514,4,0,1,0,0
2,263.35,17,1851,5,0,0,0,0
3,182.59,4,153,2,0,0,1,0
4,33.92,15,1041,2,0,0,0,0


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import joblib

X = df.drop("fraud", axis=1)
y = df["fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier()
model.fit(X_train, y_train)

print("Accuracy:", model.score(X_test, y_test))

joblib.dump(model,"fraud_model.pkl")

Accuracy: 0.899


['fraud_model.pkl']

In [3]:
!pip install streamlit
!pip install pyngrok  # Required to expose the Streamlit app

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 94.0 MB/s eta 0:00:00


In [6]:
import streamlit as st
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Load dataset
df = pd.read_csv("fraud_detection_dataset_10000.csv")

# Sidebar for navigation
st.sidebar.title("Navigation")
options = ["Fraud Detection", "Data Dashboard", "Model Insights"]
choice = st.sidebar.selectbox("Go to", options)

# Load model
rf_model = joblib.load("fraud_model.pkl")

# Fraud Detection Interface
if choice == "Fraud Detection":
    st.title("⚠ Online Transaction Fraud Detection ⚠")

    amount = st.number_input("Transaction Amount", min_value=0.0, value=100.0)
    hour = st.slider("Transaction Hour", 0, 23, 12)
    account_age = st.number_input("Account Age (days)", min_value=1, value=365)
    transactions = st.number_input("Transactions Last 24h", min_value=0, value=1)
    location = st.selectbox("Location Change", [0,1])
    device = st.selectbox("New Device", [0,1])
    prev_fraud = st.number_input("Previous Fraud Count", min_value=0, value=0)

    if st.button("Predict"):
        data = [[amount, hour, account_age, transactions, location, device, prev_fraud]]
        pred = rf_model.predict(data)
        if pred[0] == 1:
            st.error("🚨 Fraud Detected!")
        else:
            st.success("✅ Legitimate Transaction")

# Data Dashboard
elif choice == "Data Dashboard":
    st.title("📊 Fraud Analytics Dashboard")
    st.subheader("Dataset Overview")
    st.dataframe(df.head(10))

    st.subheader("Fraud vs Legitimate Transactions")
    fig1, ax1 = plt.subplots()
    df['fraud'].value_counts().plot(kind='pie', autopct='%1.1f%%', colors=['green','red'], ax=ax1)
    ax1.set_ylabel("")
    st.pyplot(fig1)

    st.subheader("Transaction Amount Distribution")
    fig2, ax2 = plt.subplots()
    sns.histplot(df['transaction_amount'], bins=50, kde=True, color='blue', ax=ax2)
    st.pyplot(fig2)

    st.subheader("Fraud by Hour of Day")
    fig3, ax3 = plt.subplots()
    df.groupby('transaction_hour')['fraud'].mean().plot(kind='bar', color='orange', ax=ax3)
    ax3.set_ylabel("Fraud Probability")
    st.pyplot(fig3)

# Model Insights
elif choice == "Model Insights":
    st.title("🧠 Model Insights")
    st.subheader("Feature Importance (Random Forest)")
    features = df.drop("fraud", axis=1).columns
    importances = rf_model.feature_importances_
    fi_df = pd.DataFrame({'Feature':features, 'Importance':importances}).sort_values(by='Importance', ascending=False)

    fig4, ax4 = plt.subplots()
    sns.barplot(x='Importance', y='Feature', data=fi_df, palette='viridis', ax=ax4)
    st.pyplot(fig4)

    st.subheader("Model Accuracy")
    X = df.drop("fraud", axis=1)
    y = df['fraud']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    st.write(f"Random Forest Accuracy: {rf_model.score(X_test, y_test)*100:.2f}%")

2026-03-08 13:59:25.297 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-08 13:59:25.475 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-03-08 13:59:25.476 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-08 13:59:25.477 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-08 13:59:25.478 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-08 13:59:25.481 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-08 13:59:25.484 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-08 13:59:25.486 Thread 'MainThread': mi

In [10]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# Load dataset
df = pd.read_csv("fraud_detection_dataset_10000.csv")

# Load trained Random Forest model
rf_model = joblib.load("fraud_model.pkl")

# Sidebar navigation
st.sidebar.title("Navigation")
options = ["Fraud Detection", "Data Dashboard", "Model Insights"]
choice = st.sidebar.selectbox("Go to", options)

# Fraud Detection Interface
if choice == "Fraud Detection":
    st.title("⚠ Online Transaction Fraud Detection ⚠")

    amount = st.number_input("Transaction Amount", min_value=0.0, value=100.0)
    hour = st.slider("Transaction Hour", 0, 23, 12)
    account_age = st.number_input("Account Age (days)", min_value=1, value=365)
    transactions = st.number_input("Transactions Last 24h", min_value=0, value=1)
    location = st.selectbox("Location Change", [0,1])
    device = st.selectbox("New Device", [0,1])
    prev_fraud = st.number_input("Previous Fraud Count", min_value=0, value=0)

    if st.button("Predict"):
        data = [[amount, hour, account_age, transactions, location, device, prev_fraud]]
        pred = rf_model.predict(data)
        if pred[0] == 1:
            st.error("🚨 Fraud Detected!")
        else:
            st.success("✅ Legitimate Transaction")

# Data Dashboard
elif choice == "Data Dashboard":
    st.title("📊 Fraud Analytics Dashboard")
    st.subheader("Dataset Preview")
    st.dataframe(df.head(10))

    st.subheader("Fraud vs Legitimate Transactions")
    fig1, ax1 = plt.subplots()
    df['fraud'].value_counts().plot(kind='pie', autopct='%1.1f%%', colors=['green','red'], ax=ax1)
    ax1.set_ylabel("")
    st.pyplot(fig1)

    st.subheader("Transaction Amount Distribution")
    fig2, ax2 = plt.subplots()
    sns.histplot(df['transaction_amount'], bins=50, kde=True, color='blue', ax=ax2)
    st.pyplot(fig2)

    st.subheader("Fraud by Hour of Day")
    fig3, ax3 = plt.subplots()
    df.groupby('transaction_hour')['fraud'].mean().plot(kind='bar', color='orange', ax=ax3)
    ax3.set_ylabel("Fraud Probability")
    st.pyplot(fig3)

# Model Insights
elif choice == "Model Insights":
    st.title("🧠 Model Insights")
    st.subheader("Feature Importance (Random Forest)")
    features = df.drop("fraud", axis=1).columns
    importances = rf_model.feature_importances_
    fi_df = pd.DataFrame({'Feature':features, 'Importance':importances}).sort_values(by='Importance', ascending=False)

    fig4, ax4 = plt.subplots()
    sns.barplot(x='Importance', y='Feature', data=fi_df, palette='viridis', ax=ax4)
    st.pyplot(fig4)

    st.subheader("Model Accuracy")
    X = df.drop("fraud", axis=1)
    y = df['fraud']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    st.write(f"Random Forest Accuracy: {rf_model.score(X_test, y_test)*100:.2f}%")

Writing app.py


In [12]:
!pip install pyngrok

In [15]:
# Step 1: Install dependencies
!pip install streamlit pyngrok scikit-learn matplotlib seaborn -q

In [16]:
# Step 2: Write the Streamlit app to app.py
%%writefile app.py
import streamlit as st
st.title("Test Streamlit GUI from Colab")
st.write("This confirms Streamlit is working!")

Overwriting app.py


In [17]:
# Step 3: Run Streamlit with ngrok
from pyngrok import ngrok
import os

# Kill previous ngrok tunnels (if any)
ngrok.kill()

# IMPORTANT: Replace 'YOUR_AUTHTOKEN' with your actual ngrok authentication token
# You can get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token("YOUR_AUTHTOKEN")

# Start Streamlit in the background
get_ipython().system_raw("streamlit run app.py &")

# Create public URL
url = ngrok.connect(port=8501)
print("Your Streamlit app is live at:", url)

ERROR:pyngrok.process.ngrok:t=2026-03-08T14:11:30+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-03-08T14:11:30+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-03-08T14:11:30+0000 lvl=eror msg="terminating with error" obj=app err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your aut

PyngrokNgrokError: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.